# Notebook D – Descriptive statistics and drift
Reproduces the descriptive numbers in the paper that do not come from a model: the effect of removing duplicates (Table 2 and Section 3.1), the non-refundable duplicates (Section 4.4), the cancellation rates by attribute (Table 9) and the change in cancellation rate and guest mix over time (Section 5.3). It uses the same cleaning as Notebooks A and B and runs on a CPU in seconds.

Put `hotel_bookings.csv` in the same folder and run all cells.

In [1]:
# Cell 1: Load data and remove exact duplicates (same steps as Notebook A)
import pandas as pd
df_raw = pd.read_csv('hotel_bookings.csv')
dup = df_raw.duplicated()
df = df_raw.drop_duplicates().reset_index(drop=True)
df['children'] = df['children'].fillna(0)
n_dedup = len(df)
df = df[(df['adults'] + df['children'] + df['babies']) > 0].reset_index(drop=True)
print(f'Raw records:                     {len(df_raw):,}')
print(f'Exact duplicates removed:        {dup.sum():,} ({dup.mean():.1%})')
print(f'Records after removing duplicates: {n_dedup:,}')
print(f'Records with zero guests removed: {n_dedup - len(df):,}')
print(f'Records used:                    {len(df):,}')
print(f'Not cancelled / cancelled:       {(df.is_canceled == 0).sum():,} / {(df.is_canceled == 1).sum():,}')
print(f'Cancellation rate before / after: {df_raw.is_canceled.mean():.1%} / {df.is_canceled.mean():.1%}')
print(f'Share of removed duplicates that were cancellations: {df_raw[dup].is_canceled.mean():.1%}')

Raw records:                     119,390
Exact duplicates removed:        31,994 (26.8%)
Records after removing duplicates: 87,396
Records with zero guests removed: 166
Records used:                    87,230
Not cancelled / cancelled:       63,221 / 24,009
Cancellation rate before / after: 37.0% / 27.5%
Share of removed duplicates that were cancellations: 63.1%


In [2]:
# Cell 2: Non-refundable bookings that were duplicates
nonref = df_raw['deposit_type'] == 'Non Refund'
print(f'Non-refundable bookings in the raw file: {nonref.sum():,}')
print(f'Of these, duplicates: {(nonref & dup).sum():,} ({(nonref & dup).sum() / nonref.sum():.0%})')
print(f'Left after removing duplicates: {(nonref & ~dup).sum():,}')

Non-refundable bookings in the raw file: 14,587
Of these, duplicates: 13,549 (93%)
Left after removing duplicates: 1,038


In [3]:
# Cell 3: Table 9 - cancellation rates by attribute in the deduplicated bookings
lead_band = pd.cut(df['lead_time'], [-1, 7, 30, 90, 180, 365, 10**5],
                   labels=['0-7 days', '8-30 days', '31-90 days', '91-180 days', '181-365 days', 'More than 365 days'])
requests = df['total_of_special_requests'].clip(upper=3).map({0: '0', 1: '1', 2: '2', 3: '3 or more'})
parking = (df['required_car_parking_spaces'] > 0).map({False: 'None', True: 'One or more'})
country = (df['country'] == 'PRT').map({True: 'Portugal', False: 'All other countries'})
groups = [('Lead time', lead_band), ('Special requests', requests), ('Parking spaces requested', parking),
          ('Market segment', df['market_segment']), ('Country', country), ('Deposit type', df['deposit_type'])]
table9 = pd.concat([df.groupby(g, observed=True)['is_canceled'].agg(Bookings='size', Rate='mean')
                    .assign(Attribute=name).reset_index(names='Group') for name, g in groups])
table9 = table9[(table9['Attribute'] != 'Market segment') | (table9['Bookings'] >= 1000)]
table9['Cancellation rate'] = (100 * table9['Rate']).round(1).astype(str) + '%'
table9 = table9[['Attribute', 'Group', 'Bookings', 'Cancellation rate']]
table9.to_csv('table_9_cancellation_rates.csv', index=False)
print(table9.to_string(index=False))
print('Market segments with fewer than 1,000 bookings are omitted.')

               Attribute               Group  Bookings Cancellation rate
               Lead time            0-7 days     18203              8.4%
               Lead time           8-30 days     16326             25.4%
               Lead time          31-90 days     22720             32.0%
               Lead time         91-180 days     18224             35.0%
               Lead time        181-365 days     11193             39.7%
               Lead time  More than 365 days       564             40.8%
        Special requests                   0     43789             33.3%
        Special requests                   1     28976             22.4%
        Special requests                   2     11795             21.3%
        Special requests           3 or more      2670             16.2%
Parking spaces requested                None     79924             30.0%
Parking spaces requested         One or more      7306              0.0%
          Market segment           Corporate      4

In [4]:
# Cell 4: Drift over time - cancellation rate and share of Portuguese guests by arrival year
drift = pd.DataFrame({'Bookings': df.groupby('arrival_date_year').size(),
                      'Cancellation rate': df.groupby('arrival_date_year')['is_canceled'].mean(),
                      'Share from Portugal': df.groupby('arrival_date_year')['country'].apply(lambda c: (c == 'PRT').mean())})
drift.to_csv('table_drift_by_year.csv')
print((drift.assign(**{'Cancellation rate': (100 * drift['Cancellation rate']).round(1).astype(str) + '%',
                       'Share from Portugal': (100 * drift['Share from Portugal']).round(1).astype(str) + '%'})).to_string())

                   Bookings Cancellation rate Share from Portugal
arrival_date_year                                                
2015                  13284             20.3%               50.6%
2016                  42313             26.5%               30.7%
2017                  31633             31.9%               24.2%
